In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def delete_file(filename: str) -> str:
    """Delete a file."""
    return f"File '{filename}' has been deleted."


model = ChatOpenAI(
    model="qwen/qwen3-vl-4b",
    base_url="http://127.0.0.1:1234/v1",
    api_key="dummy",
    temperature=0.1
)


checkpointer = InMemorySaver()


agent = create_agent(
    model=model,
    tools=[delete_file],

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "delete_file": True
            }
        )
    ],

    checkpointer=checkpointer
)


config = {
    "configurable": {
        "thread_id": "user-1"
    }
}


# STEP 1: Ask the agent
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete the file test.txt"
            }
        ]
    },
    config=config
)

print("FIRST RESPONSE:")
print(response)


# STEP 2: Human approves
response = agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config
)

print("\nAFTER APPROVAL:")
print(response)

FIRST RESPONSE:
{'messages': [HumanMessage(content='Delete the file test.txt', additional_kwargs={}, response_metadata={}, id='1fb82ddc-69a4-48b2-b4f5-18cc08f1fc25'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 149, 'total_tokens': 170, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3-vl-4b', 'system_fingerprint': 'qwen/qwen3-vl-4b', 'id': 'chatcmpl-9kg3jfmnca43wdkkqmr7k', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a043d2-824e-7a62-a272-00362c4e4ab9-0', tool_calls=[{'name': 'delete_file', 'args': {'filename': 'test.txt'}, 'id': '640209529', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 21, 'total_tokens': 170, 'input_token_details': {}, 'ou